# Mirage API server + Cloudflare quick tunnel

Run this in a fresh Google Colab **T4 GPU** runtime. Before running setup, set `REPO_URL` to the Mirage Git clone URL. This notebook runs the real backend (`MOCK=0`), waits for `/api/v1/health`, starts a temporary Cloudflare tunnel, and prints its public URL. The URL changes when the runtime restarts.

**Privacy:** the tunnel makes this API publicly reachable without authentication. Use only test photos you are comfortable sending to this temporary server, and stop the runtime when finished. Colab sessions and the tunnel are temporary.

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Set this to the repository's HTTPS or SSH clone URL before running this cell.
REPO_URL = ""
REPO_DIR = Path("/content/mirage")

if not REPO_URL.strip():
    raise ValueError("Set REPO_URL to the Mirage Git clone URL, then rerun this cell.")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if (REPO_DIR / "backend" / "app" / "main.py").is_file():
    PROJECT_DIR = REPO_DIR / "backend"
elif (REPO_DIR / "app" / "main.py").is_file():
    PROJECT_DIR = REPO_DIR
else:
    raise FileNotFoundError("The clone must contain app/main.py, either at its root or under backend/.")

requirements = PROJECT_DIR / "requirements.txt"
if not requirements.is_file():
    raise FileNotFoundError(f"Backend requirements file not found: {requirements}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
# Prevent the optional torchao/diffusers incompatibility seen in some Colab images.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)

os.environ["MOCK"] = "0"
os.environ["DEVICE"] = "auto"
os.environ["DISABLE_SIM"] = "0"
os.environ["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
print(f"Backend checkout: {PROJECT_DIR}")
print("Dependencies installed. Continue to start the server.")

In [ ]:
import time
import requests

API_BASE = "http://127.0.0.1:8000/api/v1"
SERVER_LOG_PATH = Path("/content/mirage-uvicorn.log")
server_log = SERVER_LOG_PATH.open("w")
server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=PROJECT_DIR,
    env=os.environ.copy(),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

for attempt in range(180):
    if server_process.poll() is not None:
        server_log.flush()
        raise RuntimeError(f"Uvicorn exited. Inspect {SERVER_LOG_PATH} for details.")
    try:
        health_response = requests.get(f"{API_BASE}/health", timeout=3)
        health_response.raise_for_status()
        health = health_response.json()
        print("Backend is ready:", health)
        break
    except requests.RequestException:
        time.sleep(2)
else:
    server_log.flush()
    raise TimeoutError(f"Backend did not become healthy. Inspect {SERVER_LOG_PATH}.")

In [ ]:
import os
import queue
import re
import threading
import urllib.request

cloudflared_path = Path("/content/cloudflared")
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    cloudflared_path,
)
cloudflared_path.chmod(0o755)
tunnel_process = subprocess.Popen(
    [str(cloudflared_path), "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_lines = queue.Queue()
def collect_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.put(line)

threading.Thread(target=collect_tunnel_output, daemon=True).start()
public_url = None
recent_tunnel_logs = []
deadline = time.monotonic() + 120
while time.monotonic() < deadline and tunnel_process.poll() is None:
    try:
        log_line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    recent_tunnel_logs.append(log_line.rstrip())
    recent_tunnel_logs = recent_tunnel_logs[-20:]
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_line)
    if match:
        public_url = match.group(0)
        break
if public_url is None:
    raise RuntimeError("Could not obtain a Cloudflare URL. Recent tunnel logs:\n" + "\n".join(recent_tunnel_logs))

print("Mirage API URL:", public_url)
print("Health check:", public_url + "/api/v1/health")

## Optional keep-alive

Run this cell only while recording or testing. Interrupt it to stop the periodic health check. Colab may still disconnect an idle or time-limited runtime.

In [ ]:
try:
    while True:
        response = requests.get(f"{API_BASE}/health", timeout=10)
        response.raise_for_status()
        print("Health OK", time.strftime("%H:%M:%S"))
        time.sleep(60)
except KeyboardInterrupt:
    print("Keep-alive stopped; server and tunnel remain active in this runtime.")

## Optional real-pipeline warm-up

To warm the model before a demo, upload a disposable test image here. This runs one real cloak and one simulator preset through the API and can take several minutes. The simulator requires a working CUDA runtime.

In [ ]:
from google.colab import files

warmup_upload = files.upload()
if len(warmup_upload) != 1:
    raise ValueError("Upload exactly one warm-up image.")
image_name, image_bytes = next(iter(warmup_upload.items()))
accepted = requests.post(
    f"{API_BASE}/cloak",
    files={"image": (image_name, image_bytes)},
    data={"strength": "standard", "face_focus": "true", "robust_mode": "true"},
    timeout=60,
)
accepted.raise_for_status()
cloak_job_id = accepted.json()["job_id"]

def wait_for_job(job_id, timeout_seconds=900):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        response = requests.get(f"{API_BASE}/jobs/{job_id}", timeout=15)
        response.raise_for_status()
        job = response.json()
        print(job["type"], job["status"], job["stage"], job["progress"])
        if job["status"] == "done":
            return job
        if job["status"] == "error":
            raise RuntimeError(f"Job failed: {job.get('error')}")
        time.sleep(3)
    raise TimeoutError(f"Job {job_id} did not finish in time.")

cloak_job = wait_for_job(cloak_job_id)
sim_accepted = requests.post(
    f"{API_BASE}/attack-sim",
    json={"job_id": cloak_job_id, "preset": "background_snow"},
    timeout=30,
)
sim_accepted.raise_for_status()
sim_job = wait_for_job(sim_accepted.json()["job_id"])
print("Warm-up completed; cloak and simulator jobs are ready.")